In [3]:
import pandas as pd

# Əgər yuxarıdakı kodu eyni notebook-da işləmisənsə, bu lazım deyil.
# Sadəcə df artıq var.

# Əgər faylı saxlamısansa, path-ni öz kompüterinə uyğun dəyiş:
df = pd.read_csv(
    r"azer_forest_sensors_5min_synthetic.csv"
)

print(df.head())
print(df.columns)


        lat       lon  timestep_5min_index  temp_c  humidity_pct  \
0  40.56181  46.98409                  182   28.58          51.6   
1  41.42607  47.18359                   35   17.13          67.4   
2  41.09799  48.13637                   36   29.80          42.0   
3  40.89799  46.85001                   13   21.86          53.3   
4  40.23403  48.17412                  122   26.64          26.0   

   wind_speed_ms  wind_dir_deg  solar_rad_wm2  rain_last_24h_mm  vpd_kpa  \
0           4.82         243.4         1000.0              1.75     2.38   
1           3.66         157.8          173.6              0.38     1.72   
2           4.82          16.6           87.1              2.66     2.60   
3           2.68         228.3           26.4              1.71     2.14   
4           5.61          30.9         1000.0              3.06     2.83   

   co_ppm  co2_ppm  tvoc_ppb  h2_idx  ethanol_idx  hydrocarbon_idx  \
0   0.248    442.9      58.6   12.83         4.00             6.

In [10]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

n = 20000  # number of 5-min records

# --- 1) Time ---
start_time = datetime(2024, 6, 1, 0, 0, 0)
timestamps = [start_time + timedelta(minutes=5 * i) for i in range(n)]
unix_timestamps = [int(ts.timestamp()) for ts in timestamps]
hour_of_day = np.array([ts.hour + ts.minute/60 for ts in timestamps])

# --- 2) Regions ---
regions = np.random.choice(["Shamakhi", "Gabala", "Lerik", "Guba"], size=n)

region_params = {
    "Shamakhi": {"temp": 32, "hum": 32, "wind": 5, "rain_scale": 1.0, "risk_bias": 0.6},
    "Gabala":   {"temp": 28, "hum": 40, "wind": 3, "rain_scale": 1.4, "risk_bias": 0.2},
    "Lerik":    {"temp": 24, "hum": 55, "wind": 2, "rain_scale": 2.0, "risk_bias": -0.2},
    "Guba":     {"temp": 26, "hum": 45, "wind": 4, "rain_scale": 1.3, "risk_bias": 0.1},
}

base_temp = np.array([region_params[r]["temp"] for r in regions])
base_hum = np.array([region_params[r]["hum"] for r in regions])
base_wind = np.array([region_params[r]["wind"] for r in regions])
rain_scale = np.array([region_params[r]["rain_scale"] for r in regions])
risk_bias_region = np.array([region_params[r]["risk_bias"] for r in regions])

lat_center = {
    "Shamakhi": 40.6,
    "Gabala": 40.98,
    "Lerik": 38.8,
    "Guba": 41.0,
}
lon_center = {
    "Shamakhi": 48.6,
    "Gabala": 47.85,
    "Lerik": 48.4,
    "Guba": 48.9,
}

lat = np.array([lat_center[r] for r in regions]) + np.random.normal(0, 0.05, n)
lon = np.array([lon_center[r] for r in regions]) + np.random.normal(0, 0.05, n)

# --- 3) Meteoroloji dəyişənlər ---
temp = np.random.normal(base_temp, 4)
humidity = np.clip(np.random.normal(base_hum, 15), 5, 95)
wind_speed = np.abs(np.random.normal(base_wind, 2))
wind_dir = np.random.uniform(0, 360, n)

solar_rad = np.clip(
    1000 * np.sin((np.pi / 12) * (hour_of_day - 6)),  # 6–18 arası günəş var
    0, 1000
)

rain_24h = np.abs(np.random.exponential(1.5 * rain_scale, n))

vpd = np.clip(
    2.0 + 0.03 * (temp - 25) - 0.02 * (humidity - 40),
    0.05, 6.0
)

# --- 4) Combustion/gas sensorları ---
CO = np.clip(np.random.normal(0.3, 0.2, n), 0.03, 5.0)
CO2 = np.clip(np.random.normal(420, 60, n), 350, 2500)
TVOC = np.clip(np.random.exponential(60, n), 5, 3000)
H2 = np.clip(np.random.normal(10, 7, n), 0.5, 150)
ethanol = np.clip(np.random.exponential(7, n), 0.1, 400)
hydrocarb = np.clip(np.random.exponential(7, n), 0.1, 400)

risk_score_raw = (
    0.07 * (temp - 25)
    - 0.06 * (humidity - 40)
    + 0.09 * wind_speed
    + 0.5 * (solar_rad / 1000)
    - 0.6 * (rain_24h - 1)
    + 0.45 * (vpd - 2)
    + 0.35 * (CO - 0.3)
    + 0.001 * (CO2 - 420)
    + 0.0012 * (TVOC - 60)
    + 0.018 * (H2 - 10)
    + 0.012 * (ethanol - 7)
    + 0.012 * (hydrocarb - 7)
    + risk_bias_region
)

risk_score_noisy = risk_score_raw + np.random.normal(0, 1.0, n)

threshold = np.percentile(risk_score_noisy, 60)  # ~40% positive
ignite_next_5min = (risk_score_noisy >= threshold).astype(int)

df = pd.DataFrame({
    "datetime": timestamps,
    "unix_timestamp": unix_timestamps,
    "region": regions,
    "lat": lat.round(5),
    "lon": lon.round(5),
    "temp_c": temp.round(2),
    "humidity_pct": humidity.round(1),
    "wind_speed_ms": wind_speed.round(2),
    "wind_dir_deg": wind_dir.round(1),
    "solar_rad_wm2": solar_rad.round(1),
    "rain_last_24h_mm": rain_24h.round(2),
    "vpd_kpa": vpd.round(2),
    "co_ppm": CO.round(3),
    "co2_ppm": CO2.round(1),
    "tvoc_ppb": TVOC.round(1),
    "h2_idx": H2.round(2),
    "ethanol_idx": ethanol.round(2),
    "hydrocarbon_idx": hydrocarb.round(2),
    "ignite_next_5min": ignite_next_5min,
    "risk_score_noisy": risk_score_noisy.round(3),
})

df.to_csv("azer_forest_sensors_20k_regions.csv", index=False)
print(df["ignite_next_5min"].value_counts(normalize=True))
print(df.head())


ignite_next_5min
0    0.6
1    0.4
Name: proportion, dtype: float64
             datetime  unix_timestamp    region       lat       lon  temp_c  \
0 2024-06-01 00:00:00      1717185600     Lerik  38.72561  48.43918   29.68   
1 2024-06-01 00:05:00      1717185900      Guba  40.94374  48.90038   28.80   
2 2024-06-01 00:10:00      1717186200  Shamakhi  40.61944  48.63274   30.01   
3 2024-06-01 00:15:00      1717186500     Lerik  38.74131  48.34928   29.62   
4 2024-06-01 00:20:00      1717186800     Lerik  38.85563  48.40717   25.84   

   humidity_pct  wind_speed_ms  wind_dir_deg  solar_rad_wm2  rain_last_24h_mm  \
0          51.8           0.25         133.0            0.0              0.90   
1          54.8           3.65         355.2            0.0              0.84   
2          27.8           5.45         182.4            0.0              0.98   
3          62.8           2.96          23.1            0.0              3.20   
4          53.0           3.78         274.7        

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

# Dataseti yüklə
df = pd.read_csv("azer_forest_sensors_20k_regions.csv")

feature_cols = [
    "temp_c", "humidity_pct", "wind_speed_ms", "wind_dir_deg",
    "solar_rad_wm2", "rain_last_24h_mm", "vpd_kpa",
    "co_ppm", "co2_ppm", "tvoc_ppb",
    "h2_idx", "ethanol_idx", "hydrocarbon_idx",
    "lat", "lon"
]

X = df[feature_cols]
y = df["ignite_next_5min"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

imbalance = (len(y) - y.sum()) / y.sum()

model = XGBClassifier(
    n_estimators=600,
    max_depth=10,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    scale_pos_weight=imbalance,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred_default = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("=== Default threshold (0.5) ===")
print(classification_report(y_test, y_pred_default))
print("ROC AUC:", roc_auc_score(y_test, y_proba))


=== Default threshold (0.5) ===
              precision    recall  f1-score   support

           0       0.88      0.88      0.88      2400
           1       0.82      0.82      0.82      1600

    accuracy                           0.86      4000
   macro avg       0.85      0.85      0.85      4000
weighted avg       0.86      0.86      0.86      4000

ROC AUC: 0.9340212239583333


In [12]:
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_test, y_proba)

youden_j = tpr - fpr
best_idx = youden_j.argmax()
best_threshold = thresholds[best_idx]

print("Best threshold by Youden's J:", best_threshold)

# Bu threshold ilə proqnoz
y_pred_opt = (y_proba >= best_threshold).astype(int)

print("\n=== Optimized threshold ===")
print(classification_report(y_test, y_pred_opt))



Best threshold by Youden's J: 0.28759935

=== Optimized threshold ===
              precision    recall  f1-score   support

           0       0.91      0.83      0.87      2400
           1       0.78      0.88      0.83      1600

    accuracy                           0.85      4000
   macro avg       0.84      0.86      0.85      4000
weighted avg       0.86      0.85      0.85      4000



In [30]:
import numpy as np
import pandas as pd

def predict_risk(model, sensor_reading: dict, threshold: float):
    feature_cols = [
        "temp_c", "humidity_pct", "wind_speed_ms", "wind_dir_deg",
        "solar_rad_wm2", "rain_last_24h_mm", "vpd_kpa",
        "co_ppm", "co2_ppm", "tvoc_ppb",
        "h2_idx", "ethanol_idx", "hydrocarbon_idx",
        "lat", "lon"
    ]
    x = pd.DataFrame([sensor_reading])[feature_cols]
    prob = model.predict_proba(x)[0, 1]
    alert = prob >= threshold
    return prob, alert

current = {
  "temp_c": 400.5,
  "humidity_pct": 3,
  "wind_speed_ms": 1.8,
  "wind_dir_deg": 250,
  "solar_rad_wm2": 120,
  "rain_last_24h_mm": 4.2,
  "vpd_kpa": 0.9,
  "co_ppm": 5,
  "co2_ppm": 410,
  "tvoc_ppb": 25,
  "h2_idx": 6.5,
  "ethanol_idx": 3.1,
  "hydrocarbon_idx": 4.8,
  "lat": 40.91,
  "lon": 47.88,
  "region": "Guba"
}


prob, alert = predict_risk(model, current, best_threshold)
prob_percent = prob * 100

print("P(ignite in next 5 min) =", prob_percent)
print("ALERT?", alert)


P(ignite in next 5 min) = 0.5369827151298523
ALERT? False


In [25]:
%pip install joblib



In [ ]:
import joblib

# Save model
joblib.dump(model, "fire_ignition_xgb_model.joblib")
print("Model saved as fire_ignition_xgb_model.joblib")


Model saved as fire_ignition_xgb_model.joblib


In [27]:
import json

config = {
    "feature_cols": feature_cols,
    "best_threshold": float(best_threshold)
}

with open("fire_model_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Config saved as fire_model_config.json")


Config saved as fire_model_config.json


In [ ]:
import joblib
import json
import pandas as pd

# Load model
model = joblib.load("fire_ignition_xgb_model.joblib")

# Config-i yüklə
with open("fire_model_config.json", "r") as f:
    config = json.load(f)

feature_cols = config["feature_cols"]
best_threshold = config["best_threshold"]

def predict_risk(sensor_reading: dict):
    sample = pd.DataFrame([sensor_reading])[feature_cols]
    prob = model.predict_proba(sample)[0, 1]
    alert = prob >= best_threshold
    return prob, alert
